# Introduction to Mathematical Optimization Modeling

## Objective and Prerequisites

Are you looking to learn the basics of mathematical optimization modeling? If so, then this is a great place to start. In this tutorial, we’ll walk you through the process of building a mathematical optimization model and solving a mathematical optimization problem. We’ll begin by giving you an overview of the key components of a simple mathematical optimization problem, then show you how to create a mathematical optimization model (or, to be more precise, a mixed-integer programming or MIP model) of the problem using using the Gurobi Python API, and then demonstrate how you can automatically generate an optimal solution using the Gurobi Optimizer.

This modeling tutorial is at the introductory level, where we assume that you know Python and that you have a background on a discipline that uses quantitative methods.

You may find it helpful to refer to the [documentation](https://www.gurobi.com/resources/?category-filter=documentation)
of the Gurobi Python API. This notebook is explained in detail in our series of tutorial videos on mixed-integer linear programming. You can watch these videos by clicking
[here](https://www.gurobi.com/resource/tutorial-mixed-integer-linear-programming/)

**Download the Repository** <br />
You can download the repository containing this and other examples by clicking [here](https://github.com/Gurobi/modeling-examples/archive/master.zip).

## Problem description

Consider a consulting company that has three open positions: Tester, Java Developer, and Architect. The three top candidates (resources) for the positions are: Carlos, Joe, and Monika. The consulting company administered competency tests to each candidate in order to assess their ability to perform each of the jobs. The results of these tests are called *matching scores*. Assume that only one candidate can be assigned to a job, and at most one job can be assigned to a candidate.

The problem is to determine an assignment of resources and jobs such that each job is fulfilled, each resource is assigned to at most one job, and the total matching scores of the assignments is maximized.


## Mathematical optimization

Mathematical optimization (which is also known as mathematical programming) is a declarative approach where the modeler formulates an  optimization problem that captures the key features of a complex decision problem. The Gurobi Optimizer solves the mathematical optimization problem using state-of-the-art mathematics and computer science.

A mathematical optimization model has five components:

* Sets
* Parameters
* Decision variables
* Constraints
* Objective function(s)


In [2]:
%pip install gurobipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 73.2 MB/s eta 0:00:00


The following Python code imports the Gurobi callable library and imports the ``GRB`` class into the main namespace.

In [3]:
import gurobipy as gp
from gurobipy import GRB

In [24]:
import gurobipy as gp
from gurobipy import GRB

# Data
demand_I = [0, 10000, 10000, 12000, 12000, 16000, 16000, 20000, 20000, 0, 0, 0, 0]
demand_II = [0, 6000, 7200, 8400, 10800, 10800, 12000, 12000, 12000, 0, 0, 0, 0]

T = 12          # total weeks (1..12)
W = 8           # weeks with demand

model = gp.Model("Food_Production_Planning")

# ------------------------------------------------------------------
# Variables
# ------------------------------------------------------------------
# x[s] for s=1..7
x = [0] * 9          # index 0..8, x[0]=0, x[8]=0 constant
for s in range(1, 8):
    x[s] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"x_{s}")

# yO[s], yN[s] for s=1..7
yO = [0] * 9
yN = [0] * 9
for s in range(1, 8):
    yO[s] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"yO_{s}")
    yN[s] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"yN_{s}")

# N[t] for t=1..12
N = [None] * 13
for t in range(1, 13):
    N[t] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=100, name=f"N_{t}")

# OR, OO, NR, NO for t=1..12
OR = [None] * 13
OO = [None] * 13
NR = [None] * 13
NO = [None] * 13
for t in range(1, 13):
    OR[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"OR_{t}")
    OO[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"OO_{t}")
    NR[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"NR_{t}")
    NO[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"NO_{t}")

# Inventory and backlog
I_inv = [None] * 13
II_inv = [None] * 13
B_I = [None] * 13
B_II = [None] * 13
for t in range(1, 13):
    I_inv[t] = model.addVar(vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name=f"I_inv_{t}")
    II_inv[t] = model.addVar(vtype=GRB.CONTINUOUS, lb=-GRB.INFINITY, name=f"II_inv_{t}")
    B_I[t] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"B_I_{t}")
    B_II[t] = model.addVar(vtype=GRB.CONTINUOUS, lb=0, name=f"B_II_{t}")

# ------------------------------------------------------------------
# Constraints
# ------------------------------------------------------------------
# N initial
model.addConstr(N[1] == 0)
model.addConstr(N[2] == 0)

# N dynamics
for t in range(3, 10):
    model.addConstr(N[t] == N[t-1] + x[t-2])   # t-2 in 1..7
for t in range(10, 13):
    model.addConstr(N[t] == N[t-1])

# Total trainees
model.addConstr(gp.quicksum(x[s] for s in range(1, 8)) == 50)

# Training capacity
for s in range(1, 8):
    model.addConstr(x[s] <= 3 * (yO[s] + yN[s]))

# Trainer availability and production assignment
for t in range(1, W+1):   # weeks 1..8
    o_train = (yO[t] if t <= 7 else 0) + (yO[t-1] if t-1 >= 1 else 0)
    n_train = (yN[t] if t <= 7 else 0) + (yN[t-1] if t-1 >= 1 else 0)
    model.addConstr(OR[t] + OO[t] == 50 - o_train)
    model.addConstr(NR[t] + NO[t] == N[t] - n_train)

for t in range(W+1, T+1): # weeks 9..12
    model.addConstr(OR[t] + OO[t] == 50)
    model.addConstr(NR[t] + NO[t] == N[9])   # N[9] is the final number of new workers

# Production and inventory balance
prev_I = 0
prev_II = 0
for t in range(1, T+1):
    I_prod = 400 * (OR[t] + NR[t]) + 600 * (OO[t] + NO[t])
    II_prod = 240 * (OR[t] + NR[t]) + 360 * (OO[t] + NO[t])
    model.addConstr(I_inv[t] == prev_I + I_prod - demand_I[t])
    model.addConstr(II_inv[t] == prev_II + II_prod - demand_II[t])
    prev_I = I_inv[t]
    prev_II = II_inv[t]
    # Backlog definition
    model.addConstr(B_I[t] >= -I_inv[t])
    model.addConstr(B_II[t] >= -II_inv[t])

# ------------------------------------------------------------------
# Objective
# ------------------------------------------------------------------
obj = (gp.quicksum(240 * N[t] for t in range(1, T+1)) +
       gp.quicksum(180 * OO[t] + 300 * NO[t] for t in range(1, T+1)) +
       gp.quicksum(120 * (x[t] + x[t-1]) for t in range(1, W+1)) +
       gp.quicksum(0.5 * B_I[t] + 0.6 * B_II[t] for t in range(1, T+1)))

model.setObjective(obj, GRB.MINIMIZE)

# ------------------------------------------------------------------
# Solve and output
# ------------------------------------------------------------------
model.optimize()

if model.Status == GRB.OPTIMAL:
    print(f"Optimal total cost (excluding constant 360*50*12): {model.ObjVal:.2f}")
    # Optionally print key variables here
else:
    print("No optimal solution found")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 92 rows, 129 columns and 335 nonzeros (Min)
Model fingerprint: 0x997dda21
Model has 67 linear objective coefficients
Variable types: 48 continuous, 81 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+02]
  Objective range  [5e-01, 3e+02]
  Bounds range     [1e+02, 1e+02]
  RHS range        [5e+01, 2e+04]

Presolve removed 45 rows and 26 columns
Presolve time: 0.01s
Presolved: 47 rows, 103 columns, 218 nonzeros
Variable types: 35 continuous, 68 integer (0 binary)
Found heuristic solution: objective 197560.00000

Root relaxation: objective 6.000000e+04, 36 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incum

In [28]:
import gurobipy as gp
from gurobipy import GRB

# ================= 数据准备 =================
demand_I = [10000, 10000, 12000, 12000, 16000, 16000, 20000, 20000]
demand_II = [6000, 7200, 8400, 10800, 10800, 12000, 12000, 12000]
weeks = range(1, 9)

# 初始化模型
m = gp.Model("FactoryProduction_Training_FixedWage")

# ================= 决策变量 =================
# 培训变量 (t=1..7). 第8周不安排新培训
x = m.addVars(weeks, vtype=GRB.INTEGER, name="x") # 当周开始带徒弟的老工人
y = m.addVars(weeks, vtype=GRB.INTEGER, name="y") # 当周开始受训的新工人

# 劳动力分配
S40 = m.addVars(weeks, vtype=GRB.INTEGER, name="S40") # 工作40小时的老工人
S60 = m.addVars(weeks, vtype=GRB.INTEGER, name="S60") # 工作60小时的老工人
N40 = m.addVars(weeks, vtype=GRB.INTEGER, name="N40") # 工作40小时的新工人
N60 = m.addVars(weeks, vtype=GRB.INTEGER, name="N60") # 工作60小时的新工人

# 生产时间分配与产量
H1 = m.addVars(weeks, vtype=GRB.CONTINUOUS, name="H1") # 用于生产食品I的时间
H2 = m.addVars(weeks, vtype=GRB.CONTINUOUS, name="H2") # 用于生产食品II的时间

# 库存与缺货延迟
I1 = m.addVars(weeks, vtype=GRB.CONTINUOUS, name="I1")
B1 = m.addVars(weeks, vtype=GRB.CONTINUOUS, name="B1")
I2 = m.addVars(weeks, vtype=GRB.CONTINUOUS, name="I2")
B2 = m.addVars(weeks, vtype=GRB.CONTINUOUS, name="B2")

# ================= 约束条件 =================
# 第8周不开启新培训
m.addConstr(x[8] == 0, "NoTrain_x8")
m.addConstr(y[8] == 0, "NoTrain_y8")

# 必须在第8周末前完成50名新工人的培训
m.addConstr(gp.quicksum(y[t] for t in weeks) == 50, "TotalTrainees")

for t in weeks:
    # 1名老工人最多带3名新工人
    m.addConstr(y[t] <= 3 * x[t], f"TrainerCapacity_{t}")

    # 计算第t周处于培训期内的师傅和徒弟数量（培训期为2周）
    active_trainers = x[t] + (x[t-1] if t >= 2 else 0)
    active_trainees = y[t] + (y[t-1] if t >= 2 else 0)

    # 劳动力数量约束：可用的老工人
    m.addConstr(S40[t] + S60[t] == 50 - active_trainers, f"AvailableSkilled_{t}")

    # 劳动力数量约束：可用的新工人（开始培训2周后正式上岗）
    avail_new = gp.quicksum(y[k] for k in range(1, t-1)) if t >= 3 else 0
    m.addConstr(N40[t] + N60[t] == avail_new, f"AvailableNew_{t}")

    # 工时上限约束
    total_hours = 40 * (S40[t] + N40[t]) + 60 * (S60[t] + N60[t])
    m.addConstr(H1[t] + H2[t] <= total_hours, f"MaxHours_{t}")

    # 产量计算
    P1 = 10 * H1[t]
    P2 = 6 * H2[t]

    # 库存与延迟交货平衡方程 (Inventory Balance)
    prev_I1 = I1[t-1] if t >= 2 else 0
    prev_B1 = B1[t-1] if t >= 2 else 0
    m.addConstr(I1[t] - B1[t] == prev_I1 - prev_B1 + P1 - demand_I[t-1], f"InvBal1_{t}")

    prev_I2 = I2[t-1] if t >= 2 else 0
    prev_B2 = B2[t-1] if t >= 2 else 0
    m.addConstr(I2[t] - B2[t] == prev_I2 - prev_B2 + P2 - demand_II[t-1], f"InvBal2_{t}")

# 最终交付约束：第8周末不允许有违约未交货的情况
m.addConstr(B1[8] == 0, "TermB1")
m.addConstr(B2[8] == 0, "TermB2")

# ================= 目标函数 =================
obj = gp.LinExpr()

for t in weeks:
    active_trainers = x[t] + (x[t-1] if t >= 2 else 0)
    active_trainees = y[t] + (y[t-1] if t >= 2 else 0)

    # 人工成本 (Wage Costs)
    obj += 360 * active_trainers           # 带徒弟的老工人师傅拿原标准工资
    obj += 360 * S40[t] + 540 * S60[t]     # 参与生产的老工人（正常班 + 加班）
    obj += 120 * active_trainees           # 培训期的新工人
    # 【修改处】：新工人的加班工资统一设定为 540
    obj += 240 * N40[t] + 540 * N60[t]     # 上岗的新工人（正常班 + 加班）

    # 延迟交货的违约赔偿金 (Penalty Costs)
    obj += 0.5 * B1[t] + 0.6 * B2[t]

m.setObjective(obj, GRB.MINIMIZE)

# ================= 求解 =================
m.optimize()

if m.status == GRB.OPTIMAL:
    print(f"\n最优总成本: {m.ObjVal} 元")

    # 打印简要的人员安排供检查验证
    print("\n新工人培训进度安排 (y):")
    for t in weeks:
        if y[t].X > 0:
            print(f"第 {t} 周开始培训 {y[t].X} 名新工人")
else:
    print("\n未找到最优解。")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 53 rows, 96 columns and 220 nonzeros (Min)
Model fingerprint: 0x13bb1df7
Model has 64 linear objective coefficients
Variable types: 48 continuous, 48 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [5e-01, 7e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 2e+04]

Presolve removed 12 rows and 15 columns
Presolve time: 0.00s
Presolved: 41 rows, 81 columns, 198 nonzeros
Variable types: 44 continuous, 37 integer (0 binary)

Root relaxation: objective 2.194125e+05, 71 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0 2

In [34]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import gurobipy as gp
from gurobipy import GRB

# ---------- 数据 ----------
T = 8  # 周数

# 需求量 (kg)
demand_I = [10000, 10000, 12000, 12000, 16000, 16000, 20000, 20000]
demand_II = [6000, 7200, 8400, 10800, 10800, 12000, 12000, 12000]

# 生产率 (kg/小时)
rate_I = 10.0
rate_II = 6.0

# 工资 (元/周)
w_old = 360.0      # 原熟练工
w_new = 240.0      # 新培训熟练工
w_trainee = 120.0  # 培训期间受训者

# 加班补贴 (元/额外工时)
c_ot = 9.0

# 延期赔偿 (元/kg·周)
penalty_I = 0.5
penalty_II = 0.6

# 初始熟练工人数（均为原熟练工）
S_old_initial = 50

# 需培训新工人总数
total_new_workers = 50

# 培训能力：每名培训师每两周最多培训3人
cap_per_trainer = 3

# 每周最大工时 (正常工时40，加班后最多60)
max_hours_per_worker = 60.0
normal_hours_per_worker = 40.0

# ---------- 创建模型 ----------
model = gp.Model("Food_Production_Training")

# ---------- 决策变量 ----------
# 整数变量
x = {}  # 第t周开始培训的新工人数 (t=1..7)
y = {}  # 第t周开始担任培训师的熟练工人数 (t=1..7)
S_old = {}  # 第t周初原熟练工人数 (t=1..8)
S_new = {}  # 第t周初新培训熟练工人数 (t=1..8)

for t in range(1, T+1):
    S_old[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"S_old_{t}")
    S_new[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"S_new_{t}")
for t in range(1, 8):  # 只能在前7周开始培训
    x[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"x_{t}")
    y[t] = model.addVar(vtype=GRB.INTEGER, lb=0, name=f"y_{t}")

# 连续变量
M = {}      # 生产工人数
H = {}      # 总生产工时
hI = {}     # 生产食品I的工时
hII = {}    # 生产食品II的工时
Iplus = {}  # 食品I库存
Iminus = {} # 食品I欠货
Jplus = {}  # 食品II库存
Jminus = {} # 食品II欠货
O = {}      # 加班工时（超过40*M的部分）

for t in range(1, T+1):
    M[t] = model.addVar(lb=0, name=f"M_{t}")
    H[t] = model.addVar(lb=0, name=f"H_{t}")
    hI[t] = model.addVar(lb=0, name=f"hI_{t}")
    hII[t] = model.addVar(lb=0, name=f"hII_{t}")
    Iplus[t] = model.addVar(lb=0, name=f"Iplus_{t}")
    Iminus[t] = model.addVar(lb=0, name=f"Iminus_{t}")
    Jplus[t] = model.addVar(lb=0, name=f"Jplus_{t}")
    Jminus[t] = model.addVar(lb=0, name=f"Jminus_{t}")
    O[t] = model.addVar(lb=0, name=f"O_{t}")

# 边界辅助变量 (t=0 时库存/欠货为0)
Iplus_prev = 0.0
Iminus_prev = 0.0
Jplus_prev = 0.0
Jminus_prev = 0.0

# ---------- 目标函数 ----------
# 1. 原熟练工和新熟练工的基础工资
obj = gp.quicksum( w_old * S_old[t] + w_new * S_new[t] for t in range(1, T+1) )
# 2. 培训期工资 (受训者)
# 每位受训者在开始培训的当周和下一周各领120元，即每人总培训工资240元分布在两周
# 用 (x_t + x_{t-1}) 计算第t周领取工资的受训者人数，x_0=0, x_8=0
train_wage = 0
for t in range(1, T+1):
    x_cur = x[t] if t <= 7 else 0
    x_prev = x[t-1] if t-1 >= 1 else 0
    train_wage += w_trainee * (x_cur + x_prev)
obj += train_wage
# 3. 加班补贴
obj += c_ot * gp.quicksum( O[t] for t in range(1, T+1) )
# 4. 延期赔偿
obj += gp.quicksum( penalty_I * Iminus[t] + penalty_II * Jminus[t] for t in range(1, T+1) )

model.setObjective(obj, GRB.MINIMIZE)

# ---------- 约束 ----------
# 1. 初始熟练工人数
model.addConstr( S_old[1] == S_old_initial )
model.addConstr( S_new[1] == 0 )

# 2. 工人递推关系
# 培训需两周：第t周开始培训的人在第t+2周初成为新熟练工
# 对于 t=2..8: S_new[t] = S_new[t-1] + (第t-2周开始培训的人数)
for t in range(2, T+1):
    # 第t周初新熟练工 = 上周新熟练工 + 两周前开始培训的学员
    if t-2 >= 1:
        model.addConstr( S_new[t] == S_new[t-1] + x[t-2] )
    else:
        model.addConstr( S_new[t] == S_new[t-1] )  # x_0=0, x_{-1}=0
# 原熟练工人数保持不变
for t in range(1, T):
    model.addConstr( S_old[t+1] == S_old[t] )

# 3. 生产工人数 M_t = 总熟练工 - 正在培训的培训师人数
# 正在培训的培训师 = 本周开始培训的培训师 y_t + 上周开始的 y_{t-1} (y_0=0, y_8=0)
for t in range(1, T+1):
    total_trainers = 0
    if t <= 7:
        total_trainers += y[t]
    if t-1 >= 1:
        total_trainers += y[t-1]
    model.addConstr( M[t] == (S_old[t] + S_new[t]) - total_trainers )

# 4. 培训能力约束
# 4a. 单周: 一名培训师一周最多培训3人
for t in range(1, 8):
    model.addConstr( x[t] <= cap_per_trainer * y[t] )
# 4b. 连续两周: 每名培训师在本人培训的两周内最多培训3人
for t in range(1, 7):
    model.addConstr( x[t] + x[t+1] <= cap_per_trainer * (y[t] + y[t+1]) )

# 5. 培训总人数必须达到50人
model.addConstr( gp.quicksum( x[t] for t in range(1, 8) ) == total_new_workers )
# 第8周不能开始培训 (隐含，因为x[8]未定义)

# 6. 生产工时约束
for t in range(1, T+1):
    # 总工时不能超过最大工时 (60小时/人)
    model.addConstr( H[t] <= max_hours_per_worker * M[t] )
    # 工时分解
    model.addConstr( H[t] == hI[t] + hII[t] )
    # 加班工时线性化: O[t] >= H[t] - 40 * M[t]
    model.addConstr( O[t] >= H[t] - normal_hours_per_worker * M[t] )

# 7. 产量与工时关系
for t in range(1, T+1):
    # 产量 = 生产率 * 工时 (隐含在库存平衡中，下面直接使用)
    pass

# 8. 库存与欠货平衡
for t in range(1, T+1):
    prev_I = Iplus_prev - Iminus_prev
    prev_J = Jplus_prev - Jminus_prev
    model.addConstr( Iplus[t] - Iminus[t] == prev_I + rate_I * hI[t] - demand_I[t-1] )
    model.addConstr( Jplus[t] - Jminus[t] == prev_J + rate_II * hII[t] - demand_II[t-1] )
    # 更新上一期值
    Iplus_prev = Iplus[t]
    Iminus_prev = Iminus[t]
    Jplus_prev = Jplus[t]
    Jminus_prev = Jminus[t]

# 9. 第八周末无欠货
model.addConstr( Iminus[8] == 0 )
model.addConstr( Jminus[8] == 0 )

# 可选：增加隐式约束，确保x[t]和y[t]为0当t超出范围已由变量定义限制

# ---------- 求解 ----------
model.optimize()

# ---------- 输出结果 ----------
if model.Status == GRB.OPTIMAL:
    print("\n===== 最优解找到 =====")
    print(f"最小总成本: {model.ObjVal:,.2f} 元\n")

    print("培训计划 (每周开始培训的新工人数):")
    for t in range(1, 8):
        print(f"  第{t}周: x[{t}] = {int(x[t].X)}")
    print("\n培训师安排 (每周开始担任培训师的熟练工人数):")
    for t in range(1, 8):
        print(f"  第{t}周: y[{t}] = {int(y[t].X)}")
    print("\n工人数量变化:")
    for t in range(1, 9):
        print(f"  第{t}周初: 原熟练工={int(S_old[t].X)}, 新熟练工={int(S_new[t].X)}, 合计={int(S_old[t].X+S_new[t].X)}")
    print("\n生产情况:")
    for t in range(1, 9):
        prod_I = rate_I * hI[t].X
        prod_II = rate_II * hII[t].X
        print(f"  第{t}周: 生产I={prod_I:.1f} kg, 生产II={prod_II:.1f} kg, 总工时={H[t].X:.1f}, 加班工时={O[t].X:.1f}")
    print("\n库存/欠货:")
    for t in range(1, 9):
        print(f"  第{t}周末: I库存={Iplus[t].X:.1f}, I欠货={Iminus[t].X:.1f}, II库存={Jplus[t].X:.1f}, II欠货={Jminus[t].X:.1f}")
else:
    print(f"求解未成功, 状态码: {model.Status}")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 80 rows, 102 columns and 261 nonzeros (Min)
Model fingerprint: 0x7d089703
Model has 47 linear objective coefficients
Variable types: 72 continuous, 30 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [5e-01, 4e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 2e+04]

Found heuristic solution: objective 361352.00000
Presolve removed 30 rows and 32 columns
Presolve time: 0.00s
Presolved: 50 rows, 70 columns, 209 nonzeros
Variable types: 52 continuous, 18 integer (0 binary)

Root relaxation: objective 2.194125e+05, 48 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumb

In [30]:
print(H)

{0: <gurobi.Var H[0] (value 2840.0)>, 1: <gurobi.Var H[1] (value 3000.0)>, 2: <gurobi.Var H[2] (value 3000.0)>, 3: <gurobi.Var H[3] (value 3000.0)>, 4: <gurobi.Var H[4] (value 3000.0)>, 5: <gurobi.Var H[5] (value 1980.0)>, 6: <gurobi.Var H[6] (value 1980.0)>, 7: <gurobi.Var H[7] (value 2000.0)>}


In [18]:
print(Y.values())

dict_values([<gurobi.Var Y[0] (value 7.0)>, <gurobi.Var Y[1] (value 3.0)>, <gurobi.Var Y[2] (value 3.0)>, <gurobi.Var Y[3] (value 2.0)>, <gurobi.Var Y[4] (value 1.0)>, <gurobi.Var Y[5] (value 0.0)>, <gurobi.Var Y[6] (value 1.0)>])


In [19]:
print(Z.values())

dict_values([<gurobi.Var Z[0] (value 21.0)>, <gurobi.Var Z[1] (value 9.0)>, <gurobi.Var Z[2] (value 9.0)>, <gurobi.Var Z[3] (value 6.0)>, <gurobi.Var Z[4] (value 3.0)>, <gurobi.Var Z[5] (value 0.0)>, <gurobi.Var Z[6] (value 3.0)>])


In [14]:
print(N.values())

dict_values([<gurobi.Var N[0] (value -0.0)>, <gurobi.Var N[1] (value -0.0)>, <gurobi.Var N[2] (value 21.0)>, <gurobi.Var N[3] (value 30.0)>, <gurobi.Var N[4] (value 39.0)>, <gurobi.Var N[5] (value 45.0)>, <gurobi.Var N[6] (value 47.0)>, <gurobi.Var N[7] (value 50.0)>])


In [22]:
import gurobipy as gp
from gurobipy import GRB

# ===== DATA =====
weeks = list(range(8))  # 0..7 correspond to weeks 1..8
D_I = [10000, 10000, 12000, 12000, 16000, 16000, 20000, 20000]
D_II = [6000, 7200, 8400, 10800, 10800, 12000, 12000, 12000]

penalty_I = 0.5
penalty_II = 0.6
wage_O_base = 360
wage_N_base = 240
wage_trainee_week = 120
premium_O = 180    # 540 - 360
premium_N = 300    # 540 - 240
capacity_reg = 1200
capacity_ot  = 1800
N_init = 50
N_new_total = 50

# ===== MODEL =====
m = gp.Model("FoodProductionTraining")

# ----- Integer variables -----
n_O_reg   = m.addVars(weeks, vtype=GRB.INTEGER, name="O_reg")
n_O_ot    = m.addVars(weeks, vtype=GRB.INTEGER, name="O_ot")
n_N_reg   = m.addVars(weeks, vtype=GRB.INTEGER, name="N_reg")
n_N_ot    = m.addVars(weeks, vtype=GRB.INTEGER, name="N_ot")
n_O_train = m.addVars(weeks, vtype=GRB.INTEGER, name="O_train")
n_N_train = m.addVars(weeks, vtype=GRB.INTEGER, name="N_train")
n_trainee = m.addVars(weeks, vtype=GRB.INTEGER, name="trainee")
cum_new   = m.addVars(weeks, vtype=GRB.INTEGER, name="cum_new")

# ----- Continuous variables -----
x_O_reg_I  = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="O_reg_I")
x_O_reg_II = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="O_reg_II")
x_N_reg_I  = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="N_reg_I")
x_N_reg_II = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="N_reg_II")
x_O_ot_I   = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="O_ot_I")
x_O_ot_II  = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="O_ot_II")
x_N_ot_I   = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="N_ot_I")
x_N_ot_II  = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="N_ot_II")

I_stock  = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="I_stock")
I_back   = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="I_back")
II_stock = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="II_stock")
II_back  = m.addVars(weeks, vtype=GRB.CONTINUOUS, lb=0, name="II_back")

# ===== CONSTRAINTS =====

# 1. Original skilled worker availability
for w in weeks:
    if w == 0:
        m.addConstr(n_O_reg[w] + n_O_ot[w] + n_O_train[w] <= N_init)
    else:
        m.addConstr(n_O_reg[w] + n_O_ot[w] + n_O_train[w] + n_O_train[w-1] <= N_init)

# 2. New skilled worker availability
for w in weeks:
    if w == 0:
        m.addConstr(n_N_reg[w] + n_N_ot[w] + n_N_train[w] <= cum_new[w])
    else:
        m.addConstr(n_N_reg[w] + n_N_ot[w] + n_N_train[w] + n_N_train[w-1] <= cum_new[w])

# 3. Cumulative new skilled workers
m.addConstr(cum_new[0] == 0)
m.addConstr(cum_new[1] == 0)
for w in range(2, 8):
    m.addConstr(cum_new[w] == cum_new[w-1] + n_trainee[w-2])

# 4. Training capacity (one trainer for up to 3 trainees)
for w in range(7):      # weeks 0..6 -> actual weeks 1..7
    m.addConstr(n_trainee[w] <= 3 * (n_O_train[w] + n_N_train[w]))
m.addConstr(n_trainee[7] == 0)                # week 8
m.addConstr(n_O_train[7] == 0)                # no training starts in week 8
m.addConstr(n_N_train[7] == 0)

# 5. Total new workers to train
m.addConstr(gp.quicksum(n_trainee[w] for w in range(7)) == N_new_total)

# 6. Production capacities
for w in weeks:
    m.addConstr(3 * x_O_reg_I[w] + 5 * x_O_reg_II[w] <= capacity_reg * n_O_reg[w])
    m.addConstr(3 * x_N_reg_I[w] + 5 * x_N_reg_II[w] <= capacity_reg * n_N_reg[w])
    m.addConstr(3 * x_O_ot_I[w]  + 5 * x_O_ot_II[w]  <= capacity_ot  * n_O_ot[w])
    m.addConstr(3 * x_N_ot_I[w]  + 5 * x_N_ot_II[w]  <= capacity_ot  * n_N_ot[w])

# 7. Total weekly production (helper expressions)
P_I = {}
P_II = {}
for w in weeks:
    P_I[w]  = x_O_reg_I[w] + x_N_reg_I[w] + x_O_ot_I[w] + x_N_ot_I[w]
    P_II[w] = x_O_reg_II[w] + x_N_reg_II[w] + x_O_ot_II[w] + x_N_ot_II[w]

# 8. Inventory / backlog balance
# Week 1 (index 0): initial stocks are zero
m.addConstr(I_stock[0] - I_back[0] == P_I[0] - D_I[0])
m.addConstr(II_stock[0] - II_back[0] == P_II[0] - D_II[0])
for w in range(1, 8):
    m.addConstr(I_stock[w] - I_back[w] == I_stock[w-1] - I_back[w-1] + P_I[w] - D_I[w])
    m.addConstr(II_stock[w] - II_back[w] == II_stock[w-1] - II_back[w-1] + P_II[w] - D_II[w])

# 9. All backlogs must be cleared by the end of week 8
m.addConstr(I_back[7] == 0)
m.addConstr(II_back[7] == 0)

# ===== OBJECTIVE =====
variable_cost = (
    gp.quicksum(wage_N_base * cum_new[w] for w in weeks)                     # new skilled wages
    + gp.quicksum(wage_trainee_week * 2 * n_trainee[w] for w in range(7))    # trainee total wages (120·2 = 240)
    + gp.quicksum(premium_O * n_O_ot[w] + premium_N * n_N_ot[w] for w in weeks)  # overtime premiums
    + gp.quicksum(penalty_I * I_back[w] + penalty_II * II_back[w] for w in weeks) # backlogging penalties
)

fixed_cost = N_init * wage_O_base * 8   # original workers base salary (50*360*8 = 144,000)
total_cost = variable_cost + fixed_cost

m.setObjective(total_cost, GRB.MINIMIZE)

# ===== SOLVE =====
m.optimize()

# ===== PRINT RESULTS =====
if m.status == GRB.OPTIMAL:
    print(f"\nOptimal total cost: {m.objVal:,.2f} yuan")
    print(f"  - Variable cost: {variable_cost.getValue():,.2f}")
    print(f"  - Fixed original wages: {fixed_cost:,.2f}\n")

    print("Weekly plan:")
    print("Week | O_reg O_ot N_reg N_ot O_tr N_tr Trainee | Prod_I Prod_II | Stock_I Back_I Stock_II Back_II")
    for w in weeks:
        print(f"  {w+1}  | {int(n_O_reg[w].X):4d} {int(n_O_ot[w].X):4d} "
              f"{int(n_N_reg[w].X):4d} {int(n_N_ot[w].X):4d} "
              f"{int(n_O_train[w].X):4d} {int(n_N_train[w].X):4d} "
              f"{int(n_trainee[w].X):7d} | "
              f"{P_I[w].getValue():7.1f} {P_II[w].getValue():7.1f} | "
              f"{I_stock[w].X:7.1f} {I_back[w].X:7.1f} "
              f"{II_stock[w].X:7.1f} {II_back[w].X:7.1f}")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 85 rows, 160 columns and 343 nonzeros (Min)
Model fingerprint: 0xc8917259
Model has 47 linear objective coefficients and an objective constant of 144000
Variable types: 96 continuous, 64 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [5e-01, 3e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [5e+01, 2e+04]

Presolve removed 32 rows and 42 columns
Presolve time: 0.00s
Presolved: 53 rows, 118 columns, 263 nonzeros
Variable types: 84 continuous, 34 integer (0 binary)
Found heuristic solution: objective 236640.00000

Root relaxation: objective 2.194125e+05, 59 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl

## Resource Assignment Problem
### Data
The list $R$ contains the names of the three resources: Carlos, Joe, and Monika.

The list $J$ contains the names of the job positions: Tester, Java Developer, and Architect.

$r \in R$: index and set of resources. The resource $r$ belongs to the set of resources $R$.

$j \in J$: index and set of jobs. The job $j$ belongs to the set of jobs $J$.

In [ ]:
# Resource and job sets
R = ['Carlos', 'Joe', 'Monika']
J = ['Tester', 'JavaDeveloper', 'Architect']

The ability of each resource to perform each of the jobs is listed in the following matching scores table:

![scores](https://github.com/Gurobi/modeling-examples/blob/master/milp_tutorial/util/matching_score_data.PNG?raw=1)

For each resource $r$ and job $j$, there is a corresponding matching score $s$. The matching score $s$ can only take values between 0 and 100. That is, $s_{r,j} \in [0, 100]$ for all resources $r \in R$ and jobs $j \in J$.

We use the Gurobi Python ``multidict`` function to initialize one or more dictionaries with a single statement. The function takes a dictionary as its argument. The keys represent the possible combinations of resources and jobs.


In [ ]:
# Matching score data
combinations, scores = gp.multidict({
    ('Carlos', 'Tester'): 53,
    ('Carlos', 'JavaDeveloper'): 27,
    ('Carlos', 'Architect'): 13,
    ('Joe', 'Tester'): 80,
    ('Joe', 'JavaDeveloper'): 47,
    ('Joe', 'Architect'): 67,
    ('Monika', 'Tester'): 53,
    ('Monika', 'JavaDeveloper'): 73,
    ('Monika', 'Architect'): 47
})

The following constructor creates an empty ``Model`` object “m”. We specify the model name by passing the string "RAP" as an argument. The ``Model`` object “m” holds a single optimization problem. It consists of a set of variables, a set of constraints, and the objective function.

In [ ]:
# Declare and initialize model
m = gp.Model('RAP')

## Decision variables

To solve this assignment problem, we need to identify which resource is assigned to which job. We introduce a decision variable for each possible assignment of resources to jobs. Therefore, we have 9 decision variables.

To simplify the mathematical notation of the model formulation, we define the following indices for resources and jobs:

![variables](https://github.com/Gurobi/modeling-examples/blob/master/milp_tutorial/util/decision_variables.PNG?raw=1)

For example, $x_{2,1}$ is the decision variable associated with assigning the resource Joe to the job Tester. Therefore, decision variable $x_{r,j}$ equals 1 if resource $r \in R$  is assigned to job $j \in J$, and 0 otherwise.

The ``Model.addVars()`` method creates the decision variables for a ``Model`` object.
This method returns a Gurobi ``tupledict`` object that contains the newly created variables. We supply the ``combinations`` object as the first argument to specify the variable indices. The ``name`` keyword is used to specify a name for the newly created decision variables. By default, variables are assumed to be non-negative.

In [ ]:
# Create decision variables for the RAP model
x = m.addVars(combinations, name="assign")

## Job constraints

We now discuss the constraints associated with the jobs. These constraints need to ensure that each job is filled by exactly one resource.

The job constraint for the Tester position requires that resource 1 (Carlos), resource 2 (Joe), or resource 3 (Monika) is assigned to this job. This corresponds to the following constraint.

Constraint (Tester=1)

$$
x_{1,1} + x_{2,1} + x_{3,1} = 1
$$

Similarly, the constraints for the Java Developer and Architect positions can be defined as follows.

Constraint (Java Developer = 2)

$$
x_{1,2} + x_{2,2} + x_{3,2} = 1
$$

Constraint (Architect = 3)

$$
x_{1,3} + x_{2,3} + x_{3,3} = 1
$$

The job constraints are defined by the columns of the following table.

![jobs](https://github.com/Gurobi/modeling-examples/blob/master/milp_tutorial/util/jobs_constraints.PNG?raw=1)

In general, the constraint for the job Tester can defined as follows.

$$
x_{1,1} + x_{2,1} + x_{3,1} = \sum_{r=1}^{3 } x_{r,1} =  \sum_{r \in R} x_{r,1} = 1
$$

All of the job constraints can be defined in a similarly succinct manner. For each job $j \in J$, take the summation of the decision variables over all the resources. We can write the corresponding job constraint as follows.

$$
\sum_{r \in R} x_{r,j} = 1
$$

The ``Model.addConstrs()`` method of the Gurobi/Python API defines the job constraints of the ``Model`` object “m”. This method returns a Gurobi ``tupledict`` object that contains the job constraints.
The first argument of this method, "x.sum(‘*’, j)", is the sum method and defines the LHS of the jobs constraints as follows:
For each job $j$ in the set of jobs $J$, take the summation of the decision variables over all the resources. The $==$  defines an equality constraint, and the number "1" is the RHS of the constraints.
These constraints are saying that exactly one resource should be assigned to each job.
The second argument is the name of this type of constraints.


In [ ]:
# Create job constraints
jobs = m.addConstrs((x.sum('*',j) == 1 for j in J), name='job')

## Resource constraints

The constraints for the resources need to ensure that at most one job is assigned to each resource. That is, it is possible that not all the resources are assigned.

For example, we want a constraint that requires Carlos to be assigned to at most one of the jobs: either job 1 (Tester), job 2 (Java Developer ), or job 3 (Architect). We can write this constraint as follows.

Constraint (Carlos=1)

$$
x_{1, 1} + x_{1, 2} + x_{1, 3}  \leq 1.
$$

This constraint is less or equal than 1 to allow the possibility that Carlos is not assigned to any job. Similarly, the constraints for the resources Joe and Monika can be defined as follows:

Constraint (Joe=2)

$$
x_{2, 1} + x_{2, 2} + x_{2, 3}  \leq 1.
$$

Constraint (Monika=3)

$$
x_{3, 1} + x_{3, 2} + x_{3, 3}  \leq 1.
$$

Observe that the resource constraints are defined by the rows of the following table.

![resources](https://github.com/Gurobi/modeling-examples/blob/master/milp_tutorial/util/resource_constraints.PNG?raw=1)

The constraint for the resource Carlos can be defined as follows.

$$
x_{1, 1} + x_{1, 2} + x_{1, 3} = \sum_{j=1}^{3 } x_{1,j} = \sum_{j \in J} x_{1,j} \leq 1.
$$

Again, each of these constraints can be written in a succinct manner. For each resource $r \in R$, take the summation of the decision variables over all the jobs. We can write the corresponding resource constraint as follows.

$$
\sum_{j \in J} x_{r,j} \leq  1.
$$

The ``Model.addConstrs()`` method of the Gurobi/Python API defines the resource constraints of the ``Model`` object “m”.
The first argument of this method, "x.sum(r, ‘*’)", is the sum method and defines the LHS of the resource constraints as follows: For each resource $r$ in the set of resources $R$, take the summation of the decision variables over all the jobs.
The $<=$  defines a less or equal constraints, and the number “1” is the RHS of the constraints.
These constraints are saying that each resource can be assigned to at most 1 job.
The second argument is the name of this type of constraints.


In [ ]:
# Create resource constraints
resources = m.addConstrs((x.sum(r,'*') <= 1 for r in R), name='resource')

## Objective function

The objective function is to maximize the total matching score of the assignments that satisfy the job and resource constraints.

For the Tester job, the matching score is $53x_{1,1}$, if resource Carlos is assigned, or $80x_{2,1}$, if resource Joe is assigned, or $53x_{3,1}$, if resource Monika is assigned.
Consequently, the matching score for the Tester job is as follows, where only one term in this summation will be nonzero.

$$
53x_{1,1} + 80x_{2,1} + 53x_{3,1}.
$$

Similarly, the matching scores for the Java Developer and Architect jobs are defined as follows. The matching score for the Java Developer job is:

$$
27x_{1, 2} + 47x_{2, 2} + 73x_{3, 2}.
$$

The matching score for the Architect job is:

$$
13x_{1, 3} + 67x_{2, 3} + 47x_{3, 3}.
$$

The total matching score is the summation of each cell in the following table.

![objfcn](https://github.com/Gurobi/modeling-examples/blob/master/milp_tutorial/util/objective_function.PNG?raw=1)

The goal is to  maximize the total matching score of the assignments. Therefore, the objective function is defined as follows.

\begin{equation}
\text{Maximize} \quad (53x_{1,1} + 80x_{2,1} + 53x_{3,1}) \; +
\end{equation}

\begin{equation}
\quad (27x_{1, 2} + 47x_{2, 2} + 73x_{3, 2}) \; +
\end{equation}

\begin{equation}
\quad (13x_{1, 3} + 67x_{2, 3} + 47x_{3, 3}).
\end{equation}

Each term in parenthesis in the objective function can be expressed as follows.

\begin{equation}
(53x_{1,1} + 80x_{2,1} + 53x_{3,1}) = \sum_{r \in R} s_{r,1}x_{r,1}.
\end{equation}

\begin{equation}
(27x_{1, 2} + 47x_{2, 2} + 73x_{3, 2}) = \sum_{r \in R} s_{r,2}x_{r,2}.
\end{equation}

\begin{equation}
(13x_{1, 3} + 67x_{2, 3} + 47x_{3, 3}) = \sum_{r \in R} s_{r,3}x_{r,3}.
\end{equation}

Hence, the objective function can be concisely written as:

\begin{equation}
\text{Maximize} \quad \sum_{j \in J} \sum_{r \in R} s_{r,j}x_{r,j}.
\end{equation}

The ``Model.setObjective()`` method of the Gurobi/Python API defines the objective function of the ``Model`` object “m”. The objective expression is specified in the first argument of this method.
Notice that both the matching score parameters “score” and the assignment decision variables “x” are defined over the “combinations” keys. Therefore, we use the method “x.prod(score)” to obtain the summation of the elementwise multiplication of the "score" matrix and the "x" variable matrix.
The second argument, ``GRB.MAXIMIZE``, is the optimization "sense." In this case, we want to *maximize* the total matching scores of all assignments.

In [ ]:
# Objective: maximize total matching score of all assignments
m.setObjective(x.prod(scores), GRB.MAXIMIZE)

We use the “write()” method of the Gurobi/Python API to write the model formulation to a file named "RAP.lp".

In [ ]:
# Save model for inspection
m.write('RAP.lp')

![RAP](https://github.com/Gurobi/modeling-examples/blob/master/milp_tutorial/util/RAP_lp.PNG?raw=1)

We use the “optimize( )” method of the Gurobi/Python API to solve the problem we have defined for the model object “m”.

In [ ]:
# Run optimization engine
m.optimize()

Gurobi Optimizer version 10.0.2 build v10.0.2rc0 (win64)

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 9 columns and 18 nonzeros
Model fingerprint: 0xb343b6eb
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 8e+01]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.01s
Presolved: 6 rows, 9 columns, 18 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.6000000e+32   1.800000e+31   4.600000e+02      0s
       5    1.9300000e+02   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.930000000e+02


The ``Model.getVars()`` method of the Gurobi/Python API
retrieves a list of all variables in the Model object “m”. The ``.x`` variable attribute is used to query solution values and the ``.varName`` attribute is used to query the name of the decision variables.  

In [ ]:
# Display optimal values of decision variables
for v in m.getVars():
    if v.x > 1e-6:
        print(v.varName, v.x)

# Display optimal total matching score
print('Total matching score: ', m.objVal)

assign[Carlos,Tester] 1.0
assign[Joe,Architect] 1.0
assign[Monika,JavaDeveloper] 1.0
Total matching score:  193.0


The optimal assignment is to assign:

* Carlos to the Tester job, with a matching score of 53
* Joe to the Architect job, with a matching score of 67
* Monika to the Java Developer job, with a matching score of 73.

The maximum total matching score is 193.

## Resource Assignment Problem with a budget constraint

Now, assume there is a fixed cost $C_{r,j}$ associated with assigning a resource $r \in R$ to job $j \in J$. Assume also that there is a limited budget $B$ that can be used for job assignments.

The cost of assigning Carlos, Joe, or Monika to any of the jobs is $\$1,000$ , $\$2,000$ , and $\$3,000$  respectively. The available budget is $\$5,000$.

### Data

The list $R$ contains the names of the three resources: Carlos, Joe, and Monika.
The list $J$ contains the names of the job positions: Tester, Java Developer, and Architect.

The Gurobi Python ``multidict`` function initialize two dictionaries:
* "scores" defines the matching scores for each resource and job combination.
* "costs" defines the fixed cost associated of assigning a resource to a job.



In [ ]:
# Resource and job sets
R = ['Carlos', 'Joe', 'Monika']
J = ['Tester', 'JavaDeveloper', 'Architect']

# Matching score data
# Cost is given in thousands of dollars
combinations, scores, costs = gp.multidict({
    ('Carlos', 'Tester'): [53, 1],
    ('Carlos', 'JavaDeveloper'): [27, 1],
    ('Carlos', 'Architect'): [13,1],
    ('Joe', 'Tester'): [80, 2],
    ('Joe', 'JavaDeveloper'): [47, 2],
    ('Joe', 'Architect'): [67, 2],
    ('Monika', 'Tester'): [53, 3] ,
    ('Monika', 'JavaDeveloper'): [73, 3],
    ('Monika', 'Architect'): [47, 3]
})

# Available budget (thousands of dollars)
budget = 5

The following constructor creates an empty ``Model`` object “m”. The ``Model`` object “m” holds a single optimization problem. It consists of a set of variables, a set of constraints, and the objective function.

In [ ]:
# Declare and initialize model
m = gp.Model('RAP2')

### Decision variables

The decision variable $x_{r,j}$ is 1 if $r \in R$ is assigned to job $j \in J$, and 0 otherwise.

The ``Model.addVars()`` method defines the decision variables for the model object “m”.  

Because there is a budget constraint, it is possible that not all of the jobs will be filled. To account for this, we define a new decision variable that indicates whether or not a job is filled.

Let $g_{j}$ be equal 1 if job $j \in J$ is not filled, and 0 otherwise. This variable is a gap variable that indicates that a job cannot be filled.

***Remark:*** For the previous formulation of the RAP, we defined the assignment variables as non-negative and continuous which is the default value of the ``vtype`` argument of the ``Model.addVars()`` method.
However, in this extension of the RAP, because of the budget constraint we added to the model, we need to explicitly define these variables as binary. The ``vtype=GRB.BINARY`` argument of the ``Model.addVars()`` method defines the assignment variables as binary.

In [ ]:
# Create decision variables for the RAP model
x = m.addVars(combinations, vtype=GRB.BINARY, name="assign")

# Create gap variables for the RAP model
g = m.addVars(J, name="gap")

### Job constraints

Since we have a limited budget to assign resources to jobs, it is possible that not all the jobs can be filled. For the job constraints, there are two possibilities either a resource is assigned to fill the job, or this job cannot be filled and we need to declare a gap. This latter possibility is captured by the decision variable $g_j$. Therefore, the job constraints are written as follows.

For each job $j \in J$, exactly one resource must be assigned to the job, or the corresponding $g_j$ variable must be set to 1:

$$
\sum_{r \: \in \: R} x_{r,\; j} + g_{j} = 1.
$$


In [ ]:
# Create job constraints
jobs = m.addConstrs((x.sum('*',j) + g[j]  == 1 for j in J), name='job')

### Resource constraints

The constraints for the resources need to ensure that at most one job is assigned to each resource. That is, it is possible that not all the resources are assigned. Therefore, the resource constraints are written as follows.

For each resource $r \in R$, at most one job can be assigned to the resource:

$$
\sum_{j \: \in \: J} x_{r,\; j} \leq 1.
$$

In [ ]:
# Create resource constraints
resources = m.addConstrs((x.sum(r,'*') <= 1 for r in R), name='resource')

### Budget constraint

This constraint ensures that the cost of assigning resources to fill job requirements do not exceed the budget available. The costs of assignment and budget are in thousands of dollars.

The cost of filling the Tester job is $1x_{1,1}$, if resource Carlos is assigned, or $2x_{2,1}$, if resource Joe is assigned, or $3x_{3,1}$, if resource Monika is assigned.
Consequently, the cost of filling the Tester job is as follows, where at most one term in this summation will be nonzero.

$$
1x_{1,1} + 2x_{2,1} + 3x_{3,1}.
$$

Similarly, the cost of filling the Java Developer and Architect jobs are defined as follows. The cost of filling the Java Developer job is:

$$
1x_{1, 2} + 2x_{2, 2} + 3x_{3, 2}.
$$

The cost of filling the Architect job is:

$$
1x_{1, 3} + 2x_{2, 3} + 3x_{3, 3}.
$$

Hence, the total cost of filling the jobs should be less or equal than the budget available.

\begin{equation}
(1x_{1,1} + 2x_{2,1} + 3x_{3,1}) \; +
\end{equation}

\begin{equation}
(1x_{1, 2} + 2x_{2, 2} + 3x_{3, 2}) \; +
\end{equation}

\begin{equation}
(1x_{1, 3} + 2x_{2, 3} + 3x_{3, 3}) \leq 5
\end{equation}

Each term in parenthesis in the budget constraint can be expressed as follows.

\begin{equation}
(1x_{1,1} + 2x_{2,1} + 3x_{3,1}) = \sum_{r \in R} C_{r,1}x_{r,1}.
\end{equation}

\begin{equation}
(1x_{1, 2} + 2x_{2, 2} + 3x_{3, 2}) = \sum_{r \in R} C_{r,2}x_{r,2}.
\end{equation}

\begin{equation}
(1x_{1, 3} + 2x_{2, 3} + 3x_{3, 3}) = \sum_{r \in R} C_{r,3}x_{r,3}.
\end{equation}

Therefore, the budget constraint can be concisely written as:

\begin{equation}
\sum_{j \in J} \sum_{r \in R} C_{r,j}x_{r,j} \leq B.
\end{equation}

The ``Model.addConstr()`` method of the Gurobi/Python API defines the budget constraint of the ``Model`` object “m”.
The first argument of this method, "x.prod(costs)", is the prod method and defines the LHS of the budget constraint. The $<=$ defines a less or equal constraint, and the budget amount available is the RHS of the constraint.
This constraint is saying that the total cost of assigning resources to fill jobs requirements cannot exceed the budget available.
The second argument is the name of this constraint.

In [ ]:
budget = m.addConstr((x.prod(costs) <= budget), name='budget')

## Objective function

The objective function is similar to the RAP. The first term in the objective is the total matching score of the assignments. In this extension of the RAP, it is possible that not all jobs are filled; however, we want to heavily penalize this possibility. For this purpose, we have a second term in the objective function that takes the summation of the gap variables over all the jobs and multiply it by a big penalty $M$.

Observe that the maximum value of a matching score is 100, and the value that we give to $M$ is 101. The rationale behind the value of $M$ is that having gaps heavily deteriorates the total matching scores value.

Consequently, the objective function is to maximize the total matching score of the assignments minus the penalty associated of having gap variables with a value equal to 1.

$$
\max \; \sum_{j \; \in \; J} \sum_{r \; \in \; R} s_{r,j}x_{r,j} -M \sum_{j \in J} g_{j}
$$

In [ ]:
# Penalty for not filling a job position
M = 101

In [ ]:
# Objective: maximize total matching score of assignments
# Unfilled jobs are heavily penalized
m.setObjective(x.prod(scores) - M*g.sum(), GRB.MAXIMIZE)

In [ ]:
# Run optimization engine
m.optimize()

Gurobi Optimizer version 10.0.2 build v10.0.2rc0 (win64)

CPU model: 11th Gen Intel(R) Core(TM) i7-1165G7 @ 2.80GHz, instruction set [SSE2|AVX|AVX2|AVX512]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 7 rows, 12 columns and 30 nonzeros
Model fingerprint: 0xa1231a12
Variable types: 3 continuous, 9 integer (9 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]
Presolve time: 0.00s
Presolved: 7 rows, 12 columns, 30 nonzeros
Variable types: 0 continuous, 12 integer (12 binary)
Found heuristic solution: objective 52.0000000

Root relaxation: objective 1.350000e+02, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0  135.00000    0    2   52.00000  135.00000   160%    

The definition of the objective function includes the penalty of no filling jobs. However, we are interested in the optimal total matching score value when not all the jobs are filled. For this purpose, we need to compute the total matching score value using the matching score values $s_{r,j}$ and the assignment decision variables $x_{r,j}$.

In [ ]:
# Compute total matching score from assignment variables
total_matching_score = 0
for r, j in combinations:
    if x[r, j].x > 1e-6:
        print(x[r, j].varName, x[r, j].x)
        total_matching_score += scores[r, j]*x[r, j].x

print('Total matching score: ', total_matching_score)

assign[Joe,Tester] 1.0
assign[Monika,JavaDeveloper] 1.0
Total matching score:  153.0


### Analysis

Recall that the budget is $\$5,000$, and the total  cost associated of allocating the three resources is $\$6,000$. This means that there is not enough budget to allocate the three resources we have. Consequently, the Gurobi Optimizer must choose two resources to fill the jobs demand, leave one job unfilled, and maximize the total matching scores. Notice that the two top matching scores are 80% (Joe for the Tester job) and 73% (Monika for the Java Developer job). Also, notice that the lowest score is 13% (Carlos for the Architect job). Assigning Joe to the Tester job, Monika to the Java Developer job, and nobody to the Architect job costs $\$5,000$  and yields a total matching score of 153. This is the optimal solution found by the Gurobi Optimizer.

In [ ]:
m.dispose()
gp.disposeDefaultEnv()

Freeing default Gurobi environment


Copyright © 2020 Gurobi Optimization, LLC